In [1]:
!pip install -q tiktoken

In [2]:
from pathlib import Path
import sys
import os
import torch

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1/LLMs"
)

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1/"
    "gpt2_wikitext_20k.pt"
)

OUTPUT_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_30k.pt"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"20k checkpoint not found: {CHECKPOINT_PATH}"
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/LLMs
Checkpoint: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/gpt2_wikitext_20k.pt
Checkpoint exists: True
Device: cuda
GPU: Tesla T4


In [3]:
required_files = [
    PROJECT_ROOT / "src" / "llm" / "gpt" / "model.py",
    PROJECT_ROOT / "src" / "llm" / "training" / "trainer.py",
    PROJECT_ROOT / "data" / "wikitext-103" / "train.txt",
    PROJECT_ROOT / "data" / "wikitext-103" / "validation.txt",
    CHECKPOINT_PATH,
]

for file_path in required_files:
    print(file_path, "->", file_path.exists())

if not all(file_path.exists() for file_path in required_files):
    raise FileNotFoundError(
        "One or more required files are missing."
    )

/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/LLMs/src/llm/gpt/model.py -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/LLMs/src/llm/training/trainer.py -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/LLMs/data/wikitext-103/train.txt -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/LLMs/data/wikitext-103/validation.txt -> True
/kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/gpt2_wikitext_20k.pt -> True


In [4]:
train_path = (
    PROJECT_ROOT
    / "data"
    / "wikitext-103"
    / "train.txt"
)

validation_path = (
    PROJECT_ROOT
    / "data"
    / "wikitext-103"
    / "validation.txt"
)

train_text = train_path.read_text(
    encoding="utf-8"
)

validation_text = validation_path.read_text(
    encoding="utf-8"
)

print("Training characters:", len(train_text))
print("Validation characters:", len(validation_text))

Training characters: 540624389
Validation characters: 1147070


In [5]:
import tiktoken

from llm.preprocessing.dataset import create_data_loader

tokenizer = tiktoken.get_encoding("gpt2")

BATCH_SIZE = 1
CONTEXT_LENGTH = 1024

train_loader = create_data_loader(
    text=train_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

validation_loader = create_data_loader(
    text=validation_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Training batches: 116294
Validation batches: 243


In [6]:
from llm.gpt.config import GPT_CONFIG_124M
from llm.gpt.model import GPTModel

config = GPT_CONFIG_124M.copy()
config["context_length"] = CONTEXT_LENGTH

model = GPTModel(config).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.1,
)

print("Model created.")
print("Parameters:", sum(
    parameter.numel()
    for parameter in model.parameters()
))

Model created.
Parameters: 163037184


In [7]:
from llm.training.trainer import train_model_simple

print("Training function imported.")

Training function imported.


In [8]:
variables_to_check = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "CHECKPOINT_PATH": CHECKPOINT_PATH,
    "OUTPUT_CHECKPOINT": OUTPUT_CHECKPOINT,
    "train_loader": train_loader,
    "validation_loader": validation_loader,
    "tokenizer": tokenizer,
    "model": model,
    "optimizer": optimizer,
    "device": device,
    "config": config,
}

for name, value in variables_to_check.items():
    print(name, "-> OK" if value is not None else "MISSING")

print("Input checkpoint exists:", CHECKPOINT_PATH.exists())

PROJECT_ROOT -> OK
CHECKPOINT_PATH -> OK
OUTPUT_CHECKPOINT -> OK
train_loader -> OK
validation_loader -> OK
tokenizer -> OK
model -> OK
optimizer -> OK
device -> OK
config -> OK
Input checkpoint exists: True


In [9]:
train_losses, validation_losses, tokens_seen = (
    train_model_simple(
        model=model,
        train_loader=train_loader,
        val_loader=validation_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=100,
        eval_freq=500,
        eval_iter=5,
        start_context="The history of the world",
        tokenizer=tokenizer,
        checkpoint_path=str(OUTPUT_CHECKPOINT),
        config=config,
        max_steps=30_000,
        resume_from_checkpoint=str(CHECKPOINT_PATH),
    )
)

AMP enabled: FP16

Resuming from checkpoint: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/gpt2_wikitext_20k.pt
AMP GradScaler state restored.
Resumed from step: 19999
Resumed from epoch: 5
Ep 6 (Step 020000): Train loss 2.878, Val loss 2.789
Ep 6 (Step 020500): Train loss 2.782, Val loss 2.785
Ep 6 (Step 021000): Train loss 2.791, Val loss 2.782
Ep 6 (Step 021500): Train loss 2.974, Val loss 2.783
Ep 6 (Step 022000): Train loss 2.830, Val loss 2.785
Ep 6 (Step 022500): Train loss 2.813, Val loss 2.782
Ep 6 (Step 023000): Train loss 3.067, Val loss 2.784
Ep 6 (Step 023500): Train loss 2.824, Val loss 2.786
Ep 6 (Step 024000): Train loss 2.919, Val loss 2.781
Ep 6 (Step 024500): Train loss 3.093, Val loss 2.776
Ep 6 (Step 025000): Train loss 2.864, Val loss 2.778
Ep 6 (Step 025500): Train loss 3.017, Val loss 2.775
Ep 6 (Step 026000): Train loss 2.875, Val loss 2.769
Ep 6 (Step 026500): Train loss 2.915, Val loss 2.770
Ep 6 (Step 027000): Train loss 

In [10]:
if not OUTPUT_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint was not created: {OUTPUT_CHECKPOINT}"
    )

print("Checkpoint exists:", OUTPUT_CHECKPOINT.exists())
print(
    "Checkpoint size:",
    round(OUTPUT_CHECKPOINT.stat().st_size / 1024**3, 2),
    "GB",
)

final_checkpoint = torch.load(
    OUTPUT_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print("Final global step:", final_checkpoint["global_step"])
print("Final epoch:", final_checkpoint["epoch"])
print("Final training loss:", final_checkpoint["train_loss"])
print("Final validation loss:", final_checkpoint["val_loss"])

Checkpoint exists: True
Checkpoint size: 1.87 GB
Final global step: 29999
Final epoch: 6
Final training loss: 2.895967197418213
Final validation loss: 2.757574129104614
